<a id="phasor"></a>

$$
\text{相量 (Phasor): 用一个复数表示同频率正弦量的振幅和相位} \\ \quad \\
\text{1. 由欧拉公式 } e^{j\theta} = \cos\theta + j \sin\theta \text{ (电路里用 j 表示虚数单位, 因为 i 已经表示电流):} \\
A \cos(\omega t + \varphi) = \text{Re}\left(A e^{j(\omega t + \varphi)}\right) = \text{Re}\left(\underbrace{A e^{j\varphi}}_{\text{相量 } \mathbf{V}} e^{j\omega t}\right) \\
\mathbf{V} = A e^{j\varphi} = A \angle \varphi = A\cos\varphi + j A \sin\varphi \quad (\text{模 = 振幅, 辐角 = 初相位}) \\ \quad \\
\text{2. 为什么好用: 求导变成乘以 } j\omega \\
\frac{d}{dt} e^{j\omega t} = j\omega e^{j\omega t} \text{, 而 Re 和 d/dt 可以交换顺序 (} \omega \text{ 是实数), 所以} \\
\frac{d}{dt} \text{Re}(\mathbf{V} e^{j\omega t}) = \text{Re}(j\omega \mathbf{V} e^{j\omega t}) \implies \frac{d}{dt} \longleftrightarrow \times j\omega, \quad \int dt \longleftrightarrow \div j\omega \\
j = e^{j\pi/2} \text{: 乘以 } j\omega \text{ = 振幅乘以 } \omega \text{, 相位超前 } 90^\circ \text{ (例如 } \frac{d}{dt}\cos\omega t = -\omega\sin\omega t = \omega\cos(\omega t + 90^\circ) \text{)} \\ \quad \\
\text{3. 同频率正弦量相加 = 相量相加 (Re 是线性的):} \\
\text{Re}(\mathbf{V}_1 e^{j\omega t}) + \text{Re}(\mathbf{V}_2 e^{j\omega t}) = \text{Re}((\mathbf{V}_1 + \mathbf{V}_2) e^{j\omega t}) \\
\text{微分方程变成了代数方程, 三角函数的加减变成了复数 (平面向量) 的加减}
$$

数学基础: [泰勒展开与欧拉公式](../math_basics/calculus.ipynb#taylor) · [导数](../math_basics/calculus.ipynb#derivative)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

def phasor(amp, phase_deg):
    return amp * np.exp(1j * np.deg2rad(phase_deg))

def to_time(V, w, t):
    return np.real(V * np.exp(1j * w * t))

def polar_str(z):
    return f'{abs(z):.4f} ∠ {np.rad2deg(np.angle(z)):.2f}°'

w = 2 * np.pi * 50
t = np.linspace(0, 0.04, 2001)

# 1. 相加: 3 cos(wt + 20°) + 4 cos(wt - 70°)
V1, V2 = phasor(3, 20), phasor(4, -70)
direct = 3 * np.cos(w * t + np.deg2rad(20)) + 4 * np.cos(w * t - np.deg2rad(70))
print(f'V1 + V2 = {polar_str(V1 + V2)}, 时域直接相加 == 相量相加? {np.allclose(direct, to_time(V1 + V2, w, t))}')

# 2. 求导 = 乘以 jw
x = 2 * np.cos(w * t + np.deg2rad(30))
dx_numeric = np.gradient(x, t)
dx_phasor = to_time(1j * w * phasor(2, 30), w, t)
print(f'相量 jw V = {polar_str(1j * w * phasor(2, 30))} (振幅 x w, 相位 +90°)')
print(f'数值导数 == Re(jw V e^(jwt)) ? {np.allclose(dx_numeric[1:-1], dx_phasor[1:-1], rtol=1e-3, atol=1e-2)}')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for V, name in [(V1, 'V1'), (V2, 'V2'), (V1 + V2, 'V1 + V2')]:
    axs[0].annotate('', xy=(V.real, V.imag), xytext=(0, 0), arrowprops=dict(arrowstyle='->', lw=1.5))
    axs[0].text(V.real, V.imag, name)
axs[0].set_xlim(-1, 6)
axs[0].set_ylim(-5, 2)
axs[0].set_aspect('equal')
axs[0].set_title('phasors (complex plane)')
axs[1].plot(t * 1000, to_time(V1, w, t), label='V1')
axs[1].plot(t * 1000, to_time(V2, w, t), label='V2')
axs[1].plot(t * 1000, direct, 'k', label='sum')
axs[1].set_xlabel('t (ms)')
plot_nicely()
plt.show()

<a id="impedance"></a>

$$
\text{阻抗 (Impedance): 相量形式的欧姆定律 } \mathbf{V} = Z \mathbf{I} \text{, } Z = \frac{\mathbf{V}}{\mathbf{I}} \text{ (复数, 单位 } \Omega \text{)} \\ \quad \\
\text{由元件方程推导 (把 } \frac{d}{dt} \text{ 换成 } j\omega \text{):} \\
\text{电阻: } v = R i \implies \mathbf{V} = R \mathbf{I} \implies Z_R = R \quad (\text{电压电流同相}) \\
\text{电感: } v = L \frac{di}{dt} \implies \mathbf{V} = j\omega L \mathbf{I} \implies Z_L = j\omega L \quad (\text{电压超前电流 } 90^\circ \text{, 频率越高阻抗越大}) \\
\text{电容: } i = C \frac{dv}{dt} \implies \mathbf{I} = j\omega C \mathbf{V} \implies Z_C = \frac{1}{j\omega C} = -\frac{j}{\omega C} \quad (\text{电流超前电压 } 90^\circ \text{, 频率越高阻抗越小}) \\ \quad \\
Z = R + jX \text{: R 电阻 (Resistance), X 电抗 (Reactance), } X_L = \omega L, \ X_C = -\frac{1}{\omega C} \\
\text{导纳 (Admittance) } Y = \frac{1}{Z} \\
\text{极限情况: 直流 } \omega = 0 \text{: } Z_L = 0 \text{ (短路), } Z_C = \infty \text{ (开路), 和 RC/RL 的稳态结论一致}
$$

In [ ]:
R, L, Cap = 100.0, 0.2, 20e-6
for f in [10, 50, 500]:
    wf = 2 * np.pi * f
    print(f'f = {f:>3} Hz: Z_R = {R:.1f}, Z_L = {polar_str(1j * wf * L)}, Z_C = {polar_str(1 / (1j * wf * Cap))}')

# 用时域验证电容的阻抗: 给电容加 v = 10 cos(wt), 电流 i = C dv/dt
wf = 2 * np.pi * 50
t = np.linspace(0, 0.04, 4001)
v_c = 10 * np.cos(wf * t)
i_c = Cap * np.gradient(v_c, t)
I_phasor = phasor(10, 0) / (1 / (1j * wf * Cap))
print(f'电容电流相量 = {polar_str(I_phasor)} (超前 90°), 时域 i = C dv/dt 匹配? {np.allclose(i_c[1:-1], to_time(I_phasor, wf, t)[1:-1], atol=1e-4)}')

<a id="phasor-kirchhoff"></a>

$$
\text{相量形式的基尔霍夫定律} \\
\text{KVL: } \sum_k v_k(t) = 0 \ \forall t \implies \text{Re}\left(\left(\sum_k \mathbf{V}_k\right) e^{j\omega t}\right) = 0 \ \forall t \\
\text{取 } t = 0 \text{ 得实部为 0, 取 } \omega t = \frac{\pi}{2} \text{ 得 } \text{Re}(j \sum \mathbf{V}_k) = -\text{Im}(\sum \mathbf{V}_k) = 0 \implies \sum_k \mathbf{V}_k = 0 \\
\text{同理 KCL: } \sum_k \mathbf{I}_k = 0 \\ \quad \\
\text{所以直流电路的所有方法 (串并联, 分压, 节点法 } G v = i \text{, 戴维南) 都可以直接用于交流稳态,} \\
\text{只要把电阻换成复数阻抗, 电导换成复数导纳, 解复数线性方程组} \\ \quad \\
\text{例: } \mathbf{V}_s = 10\angle 0^\circ \text{ (f = 50 Hz)} \xrightarrow{R_1} \text{节点 A}, \ L: A \to \text{地}, \ R_2: A \to B, \ C: B \to \text{地} \\
\text{节点 A: } \frac{\mathbf{V}_A - \mathbf{V}_s}{R_1} + \frac{\mathbf{V}_A}{j\omega L} + \frac{\mathbf{V}_A - \mathbf{V}_B}{R_2} = 0 \\
\text{节点 B: } \frac{\mathbf{V}_B - \mathbf{V}_A}{R_2} + j\omega C \, \mathbf{V}_B = 0 \\
\begin{bmatrix} \frac{1}{R_1} + \frac{1}{j\omega L} + \frac{1}{R_2} & -\frac{1}{R_2} \\ -\frac{1}{R_2} & \frac{1}{R_2} + j\omega C \end{bmatrix}
\begin{bmatrix} \mathbf{V}_A \\ \mathbf{V}_B \end{bmatrix} = \begin{bmatrix} \frac{\mathbf{V}_s}{R_1} \\ 0 \end{bmatrix} \\ \quad \\
\text{验证: 时域里直接解这个电路的 ODE (状态: 电感电流 } i_L \text{, 电容电压 } v_B \text{), 等暂态衰减后} \\
\text{把稳态波形拟合成 } a\cos\omega t - b\sin\omega t = \text{Re}((a + jb) e^{j\omega t}) \text{, 得到相量 } a + jb \text{ (最小二乘)}
$$

数学基础: [线性方程组](../math_basics/linear_algebra.ipynb#inverse) · [投影矩阵与最小二乘](../math_basics/linear_algebra.ipynb#projection) · 直流版本: [节点电压法](circuit_analysis.ipynb#nodal)

In [ ]:
Vs_amp, f = 10.0, 50.0
wf = 2 * np.pi * f
R1, R2, L, Cap = 50.0, 100.0, 0.2, 20e-6

# 1. 相量法: 复数节点方程
Y = np.array([[1 / R1 + 1 / (1j * wf * L) + 1 / R2, -1 / R2],
              [-1 / R2, 1 / R2 + 1j * wf * Cap]])
VA, VB = np.linalg.solve(Y, [Vs_amp / R1, 0])
print(f'相量法: V_A = {polar_str(VA)}, V_B = {polar_str(VB)}')

# 2. 时域 ODE
def vA_from_state(t, iL, vB):
    vs = Vs_amp * np.cos(wf * t)
    return (vs / R1 - iL + vB / R2) / (1 / R1 + 1 / R2)     # 节点 A 的 KCL (代数关系)

def rhs(t, s):
    iL, vB = s
    vA = vA_from_state(t, iL, vB)
    return [vA / L, (vA - vB) / (R2 * Cap)]                   # L di/dt = v_A, C dv_B/dt = (v_A - v_B)/R2

T_end = 0.5
t_eval = np.linspace(T_end - 0.06, T_end, 3001)               # 最后 3 个周期
sol = solve_ivp(rhs, [0, T_end], [0.0, 0.0], t_eval=t_eval, rtol=1e-10, atol=1e-12, max_step=1e-4)
vA_t = vA_from_state(sol.t, sol.y[0], sol.y[1])
vB_t = sol.y[1]

def fit_phasor(x, t, w):
    M = np.column_stack([np.cos(w * t), -np.sin(w * t)])
    a, b = np.linalg.lstsq(M, x, rcond=None)[0]
    return a + 1j * b

VA_ode, VB_ode = fit_phasor(vA_t, sol.t, wf), fit_phasor(vB_t, sol.t, wf)
print(f'时域 ODE: V_A = {polar_str(VA_ode)}, V_B = {polar_str(VB_ode)}')
print(f'一致? {np.isclose(VA, VA_ode, atol=1e-5) and np.isclose(VB, VB_ode, atol=1e-5)}')

plt.figure(figsize=(4, 3))
plt.plot(sol.t * 1000, Vs_amp * np.cos(wf * sol.t), 'gray', alpha=0.5, label='v_s')
plt.plot(sol.t * 1000, vA_t, label='v_A (ODE)')
plt.plot(sol.t * 1000, to_time(VA, wf, sol.t), '--', label='v_A (phasor)')
plt.plot(sol.t * 1000, vB_t, label='v_B (ODE)')
plt.plot(sol.t * 1000, to_time(VB, wf, sol.t), '--', label='v_B (phasor)')
plt.xlabel('t (ms)')
plot_nicely()
plt.show()

<a id="complex-power"></a>

$$
\text{复功率 (Complex Power)} \\
\text{用有效值相量 } \mathbf{V} = V_{rms} e^{j\theta_v}, \ \mathbf{I} = I_{rms} e^{j\theta_i} \text{ (振幅相量除以 } \sqrt{2} \text{), 定义} \\
S = \mathbf{V} \mathbf{I}^* = V_{rms} I_{rms} e^{j(\theta_v - \theta_i)} = V_{rms} I_{rms} \cos\varphi + j V_{rms} I_{rms} \sin\varphi = P + jQ, \quad \varphi = \theta_v - \theta_i \\
\text{实部 = 有功功率 P, 虚部 = 无功功率 Q, 模 = 视在功率 |S|, 和 AC_DC 里用积化和差推出来的结果一致} \\
\text{(取共轭是为了让相位相减: 只有电压和电流的相位差才有物理意义)} \\ \quad \\
\text{对一个阻抗 } Z = R + jX \text{: } S = Z \mathbf{I} \mathbf{I}^* = |\mathbf{I}|^2 (R + jX) \implies P = I_{rms}^2 R, \ Q = I_{rms}^2 X \\
\text{有功功率只消耗在电阻上, 电感 (X > 0) 吸收无功, 电容 (X < 0) 发出无功} \\ \quad \\
\text{功率因数校正 (Power Factor Correction): 感性负载 (电动机) Q > 0, 并联电容提供 } -Q \\
\text{电容的复功率: } S_C = \mathbf{V} (j\omega C \mathbf{V})^* = \mathbf{V}\mathbf{V}^* (-j\omega C) = -j\omega C V_{rms}^2 \implies Q_C = -\omega C V_{rms}^2 \\
Q + Q_C = 0 \implies C = \frac{Q}{\omega V_{rms}^2} \text{, 此时总无功为 0, PF = 1, 有功功率不变而线路电流变小}
$$

应用: [平均功率与功率因数 (时域推导)](AC_DC.ipynb#ac-power)

In [ ]:
# 节点 A 处电源送出的电流和复功率 (用有效值)
Is = (Vs_amp - VA) / R1
Vs_rms, Is_rms = Vs_amp / np.sqrt(2), Is / np.sqrt(2)
S = Vs_rms * np.conj(Is_rms)
print(f'电源复功率 S = P + jQ = {S.real:.6f} + j{S.imag:.6f}, PF = {S.real / abs(S):.4f}')

# 时域平均功率 (最后 3 个完整周期)
vs_t = Vs_amp * np.cos(wf * sol.t)
is_t = (vs_t - vA_t) / R1
print(f'时域 mean(v i) = {np.mean(vs_t[:-1] * is_t[:-1]):.6f} W (== P)')

# 有功功率 = 各电阻消耗之和
P_R1 = abs(Is / np.sqrt(2))**2 * R1
P_R2 = abs((VA - VB) / R2 / np.sqrt(2))**2 * R2
print(f'电阻消耗: R1 = {P_R1:.6f} W, R2 = {P_R2:.6f} W, 合计 = {P_R1 + P_R2:.6f} W')

# 功率因数校正: 220 V / 50 Hz 电动机, 等效 R = 20, X_L = 15
V_line = 220.0
Z_motor = 20 + 15j
I_motor = V_line / Z_motor
S_motor = V_line * np.conj(I_motor)
C_pfc = S_motor.imag / (wf * V_line**2)
I_total = I_motor + V_line * 1j * wf * C_pfc
S_total = V_line * np.conj(I_total)
print(f'电动机: P = {S_motor.real:.1f} W, Q = {S_motor.imag:.1f} var, PF = {S_motor.real / abs(S_motor):.4f}, 电流 = {abs(I_motor):.3f} A')
print(f'并联 C = Q / (w V^2) = {C_pfc * 1e6:.2f} uF 后: P = {S_total.real:.1f} W, Q = {S_total.imag:.2e} var, PF = {S_total.real / abs(S_total):.4f}, 电流 = {abs(I_total):.3f} A')